# 01 — Core & Data Layer

Covers the building blocks available in Phases 1–4:
- `MonitoringPlan` + `TabularSchema` + `MetricSpec`
- YAML serialisation round-trip
- `DataFrameSource.load(plan)` — automatic column projection
- Sampling strategies (`LastNRows`, `TimeWindow`)
- Partitioning (`TimeBasedPartitioner`)
- `CsvSource` — file-backed source with explicit backend selection
- `ExcelSource` — Excel file source (requires `pip install ayn-ml[excel]`)

No runner yet — that comes in the next notebook.

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Synthetic dataset

In [2]:
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 500

# timestamps spread over 60 days
base = datetime(2024, 5, 1, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=int(h)) for h in rng.integers(0, 60 * 24, size=n)]
timestamps.sort()

df = pd.DataFrame(
    {
        "timestamp": timestamps,
        "y_true": rng.integers(0, 2, size=n),
        "y_pred": rng.integers(0, 2, size=n),
        "y_pred_proba": rng.uniform(0.1, 0.9, size=n),
        "age": rng.integers(20, 70, size=n).astype(float),
        "income": rng.integers(20_000, 120_000, size=n).astype(float),
        "tenure": rng.integers(0, 10, size=n),  # int-encoded categorical
        "region": rng.choice(["north", "south", "east", "west"], size=n),
    }
)

print(df.shape)
df.head()

(500, 8)


,timestamp,y_true,y_pred,y_pred_proba,age,income,tenure,region
0,2024-05-01 07:00:00+00:00,0,1,0.308279,54.0,97803.0,0,south
1,2024-05-01 09:00:00+00:00,1,1,0.357992,24.0,87185.0,6,west
2,2024-05-01 10:00:00+00:00,1,1,0.293986,57.0,112993.0,3,north
3,2024-05-01 14:00:00+00:00,1,1,0.483891,25.0,116058.0,2,south
4,2024-05-01 20:00:00+00:00,0,0,0.646607,22.0,90490.0,0,west


## 2. MonitoringPlan

In [3]:
from ayn_ml import MetricSpec, MonitoringPlan, TabularSchema

schema = TabularSchema(
    timestamp_col="timestamp",
    label_col="y_true",
    prediction_col="y_pred",
    proba_col="y_pred_proba",
    feature_types={"tenure": "categorical"},  # override int-encoded column
    model_id_col=None,
    model_version_col=None,
)

plan = MonitoringPlan(
    name="churn_monitor",
    model_id="churn_v1",
    model_version="1.0",
    data_schema=schema,
    metrics=[
        MetricSpec(name="accuracy", threshold=0.80, upper_bound=False),
        MetricSpec(name="f1", threshold=0.75, upper_bound=False),
        MetricSpec(name="psi", feature_name="age", threshold=0.2),
        MetricSpec(name="psi", feature_name="income", threshold=0.2),
        MetricSpec(name="mean", feature_name="age"),
    ],
    enable_profiling=True,  # attach a column profile to every MonitoringReport
)

print(f"Plan: {plan.name}  |  model: {plan.model_id} v{plan.model_version}")
print(f"Metrics: {[s.name for s in plan.metrics]}")
print(f"Enable profiling: {plan.enable_profiling}")

Plan: churn_monitor  |  model: churn_v1 v1.0
Metrics: ['accuracy', 'f1', 'psi', 'psi', 'mean']
Enable profiling: True


## 3. YAML serialisation round-trip

In [4]:
import yaml

# serialise to dict then to YAML string
plan_dict = plan.model_dump()
plan_yaml = yaml.dump(plan_dict, default_flow_style=False, sort_keys=False)
print(plan_yaml)

name: churn_monitor
model_id: churn_v1
model_version: '1.0'
data_schema:
  timestamp_col: timestamp
  model_id_col: null
  model_version_col: null
  type: tabular
  label_col: y_true
  prediction_col: y_pred
  proba_col: y_pred_proba
  feature_types:
    tenure: categorical
  protected_cols: null
metrics:
- name: accuracy
  metric_type: null
  feature_name: null
  params: {}
  threshold: 0.8
  upper_bound: false
- name: f1
  metric_type: null
  feature_name: null
  params: {}
  threshold: 0.75
  upper_bound: false
- name: psi
  metric_type: null
  feature_name: age
  params: {}
  threshold: 0.2
  upper_bound: true
- name: psi
  metric_type: null
  feature_name: income
  params: {}
  threshold: 0.2
  upper_bound: true
- name: mean
  metric_type: null
  feature_name: age
  params: {}
  threshold: null
  upper_bound: true
enable_profiling: true
window: null
sampling: null
description: ''



In [5]:
# round-trip: dict → MonitoringPlan
plan2 = MonitoringPlan.model_validate(plan_dict)
assert plan2.name == plan.name
assert len(plan2.metrics) == len(plan.metrics)
print("Round-trip OK")

Round-trip OK


## 4. DataFrameSource — column projection

In [6]:
from ayn_ml.data.source import DataFrameSource, required_columns

# see which columns the plan needs
needed = required_columns(plan)
print("Required columns:", needed)

# project — drops columns no metric or schema references
df_projected = DataFrameSource(df).load(plan)
print("\nProjected shape:", df_projected.shape)
print("Columns kept:", list(df_projected.columns))
print("'income' kept (metric target):", "income" in df_projected.columns)
print("'region' dropped (not referenced by schema or metrics):", "region" not in df_projected.columns)

Required columns: ['timestamp', 'y_true', 'y_pred', 'y_pred_proba', 'age', 'income']

Projected shape: (500, 6)
Columns kept: ['timestamp', 'y_true', 'y_pred', 'y_pred_proba', 'age', 'income']
'income' kept (metric target): True
'region' dropped (not referenced by schema or metrics): True


## 5. Sampling

In [7]:
from ayn_ml.data.sampling import LastNRowsSampling, TimeWindowSampling

# last 100 rows
window_last_n = LastNRowsSampling(n=100).sample(df_projected, schema)
print("LastNRowsSampling(100):", len(window_last_n), "rows")

# time window: June 2024
start = datetime(2024, 6, 1, tzinfo=timezone.utc)
end = datetime(2024, 6, 30, tzinfo=timezone.utc)
window_time = TimeWindowSampling(start, end).sample(df_projected, schema)
print(f"TimeWindowSampling(Jun 2024): {len(window_time)} rows")

LastNRowsSampling(100): 100 rows
TimeWindowSampling(Jun 2024): 235 rows


## 6. Partitioning

In [8]:
from ayn_ml.data.partitioner import TimeBasedPartitioner

# split on June 15: rows before → reference, rows after → current
cutoff = datetime(2024, 6, 15, tzinfo=timezone.utc)
current, reference = TimeBasedPartitioner(
    cutoff=cutoff,
    reference_window=timedelta(days=14),
).partition(window_time, schema)

print(f"Current (after {cutoff.date()}):    {len(current)} rows")
print(f"Reference (14 days before cutoff): {len(reference)} rows")
print(f"Total: {len(current) + len(reference)} (= {len(window_time)} in window)")

Current (after 2024-06-15):    125 rows
Reference (14 days before cutoff): 110 rows
Total: 235 (= 235 in window)


## 7. Plan-owned window and sampling configs

Window and sampling strategies can be declared on the plan so they serialise to YAML. Partitioning is applied at runtime — instantiate  directly.

In [9]:
from ayn_ml import RandomSamplingConfig, TimeWindowConfig
from ayn_ml.data.partitioner import TimeBasedPartitioner

# Plan-owned window and sampling configs serialise to YAML
plan_with_split = MonitoringPlan(
    name="churn_monitor_split",
    model_id="churn_v1",
    model_version="1.0",
    data_schema=schema,
    metrics=[MetricSpec(name="accuracy")],
    window=TimeWindowConfig(
        start=datetime(2024, 6, 1, tzinfo=timezone.utc),
        end=datetime(2024, 6, 30, tzinfo=timezone.utc),
    ),
    sampling=RandomSamplingConfig(n=200, seed=42),
)

print("window type:", plan_with_split.window.type)
print("sampling type:", plan_with_split.sampling.type)

# DataPartitioner is instantiated directly at runtime
cutoff2 = datetime(2024, 6, 15, tzinfo=timezone.utc)
df_proj2 = DataFrameSource(df).load(plan_with_split)
s = plan_with_split.window
window2 = TimeWindowSampling(s.start, s.end).sample(df_proj2, schema)
current2, reference2 = TimeBasedPartitioner(
    cutoff=cutoff2,
    reference_window=timedelta(days=14),
).partition(window2, schema)
print(f"current: {len(current2)}, reference: {len(reference2)}")

window type: time_window
sampling type: random
current: 125, reference: 110


## 8. CsvSource — load from a CSV file

`CsvSource` reads a CSV file from disk and projects to the columns required by
the plan — identical contract to `DataFrameSource`, but file-backed.

`backend` selects the engine: `"polars"`, `"pandas"`, or `"auto"` (Polars first,
pandas fallback).  `separator` is a first-class field normalised by narwhals
across all backends.  Any extra options go in `read_kwargs`, matched to the
chosen backend's API.

In [ ]:
import tempfile
from pathlib import Path

from ayn_ml.data.csv import CsvSource

# write the in-memory DataFrame to a temporary CSV file
tmp_csv = Path(tempfile.mkstemp(suffix=".csv")[1])
df.to_csv(tmp_csv, index=False)
print("CSV written to:", tmp_csv)

# load it back — backend="auto" selects Polars if installed, else pandas
csv_source = CsvSource(path=tmp_csv)
df_from_csv = csv_source.load(plan)
print("Backend type :", type(df_from_csv).__module__.split(".")[0])
print("Loaded shape :", df_from_csv.shape)
print("Columns      :", list(df_from_csv.columns))

# pipe-separated file example (separator is first-class, works on all backends)
tmp_pipe = Path(tempfile.mkstemp(suffix=".csv")[1])
df.to_csv(tmp_pipe, sep="|", index=False)
df_pipe = CsvSource(path=tmp_pipe, separator="|").load(plan)
print("\nPipe-separated shape:", df_pipe.shape)

# clean up
tmp_csv.unlink()
tmp_pipe.unlink()
print("Temp files removed.")

## 9. ExcelSource — load from an Excel file

`ExcelSource` reads a single worksheet from an `.xlsx` or `.xls` file.  It
follows the same backend pattern as `CsvSource`: `backend="polars"` uses
`fastexcel` (calamine engine, no Java), `backend="pandas"` uses `openpyxl`.

`sheet_name` is a first-class field — pass a sheet name (string) or a 0-based
integer index.  Both are translated correctly for the active backend.

Requires: `pip install ayn-ml[excel]`

In [ ]:
try:
    import openpyxl  # noqa: F401
    from ayn_ml.data.excel import ExcelSource

    # Excel does not support tz-aware datetimes — strip tz before writing
    df_excel = df.copy()
    df_excel["timestamp"] = df_excel["timestamp"].dt.tz_localize(None)

    # write to a temporary Excel file
    tmp_xlsx = Path(tempfile.mkstemp(suffix=".xlsx")[1])
    df_excel.to_excel(tmp_xlsx, sheet_name="predictions", index=False)
    print("Excel written to:", tmp_xlsx)

    # load by sheet name — backend="auto" prefers Polars (fastexcel) if installed
    excel_source = ExcelSource(path=tmp_xlsx, sheet_name="predictions")
    df_from_xlsx = excel_source.load(plan)
    print("Backend type :", type(df_from_xlsx).__module__.split(".")[0])
    print("Loaded shape :", df_from_xlsx.shape)
    print("Columns      :", list(df_from_xlsx.columns))

    # explicit pandas backend with a native pandas kwarg
    df_xlsx_pd = ExcelSource(
        path=tmp_xlsx,
        backend="pandas",
        sheet_name="predictions",
        read_kwargs={"usecols": list(df_from_xlsx.columns)},
    ).load(plan)
    print("\npandas backend shape:", df_xlsx_pd.shape)

    # clean up
    tmp_xlsx.unlink()
    print("Temp file removed.")

except ImportError:
    print("openpyxl not installed — skipping ExcelSource demo.")
    print("Install with: pip install ayn-ml[excel]")